# Block 01 – Your first agent, one cell at a time

**You'll see:** what `create_agent` builds, what goes in and out, the raw content blocks vs `.text`, and the token count.

Select the `.venv` Python as the kernel (top right in VS Code), then run cells top to bottom.

In [ ]:
# Setup: run this cell first. It finds the repo, makes it importable and loads .env
import os, sys, pathlib

def find_repo_root():
    starts = []
    if "__vsc_ipynb_file__" in globals():                 # VS Code tells us where this notebook is
        starts.append(pathlib.Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
    starts.append(pathlib.Path.cwd().resolve())
    for start in starts:
        for folder in [start, *start.parents]:
            if (folder / "common" / "config.py").exists():
                return folder
            if (folder / "agents-weekend" / "common" / "config.py").exists():
                return folder / "agents-weekend"
    raise RuntimeError(f"Can't find the agents-weekend repo from {starts}. "
                       "Open the agents-weekend folder in VS Code, or set ROOT by hand.")

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

from common.config import get_llm, get_cheap_llm, DEFAULT_MODEL, CHEAP_MODEL
from common.tracing import langfuse_config, trace_attributes, flush, tracing_enabled
print("main model:", DEFAULT_MODEL, "| cheap model:", CHEAP_MODEL, "| Langfuse:", tracing_enabled())

## 1. Build the agent
Nothing is sent to Claude yet. `create_agent` only builds a graph.

In [ ]:
from langchain.agents import create_agent
from common.helpdesk import SYSTEM_PROMPT

agent = create_agent(model=get_llm(), tools=[], system_prompt=SYSTEM_PROMPT, name="helpdesk-v1")
print(type(agent).__name__)           # a LangGraph graph, even though we imported from langchain
print(agent.get_graph().draw_mermaid())

## 2. Ask one question
The input is the **state**: a dict with a `messages` list.

In [ ]:
question = "My VPN keeps disconnecting. What should I do?"
with trace_attributes(tags=["notebook-01"], user_id="notebook"):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=langfuse_config())

for i, m in enumerate(result["messages"]):
    print(i, type(m).__name__)

## 3. Raw content blocks vs `.text`
Newer Claude models think first. `.content` is a list of blocks; `.text` is just the answer.

In [ ]:
reply = result["messages"][-1]
print("block types:", [b["type"] for b in reply.content] if isinstance(reply.content, list) else "plain string")
print()
print(reply.text)

## 4. Tokens
Thinking tokens are billed even though their text is hidden. Compare output tokens with the visible length.

In [ ]:
usage = reply.usage_metadata
print(usage)
print("visible answer ~", len(reply.text) // 4, "tokens  vs  output_tokens =", usage["output_tokens"])

## 5. Send the trace
Then open Langfuse → Tracing → filter tag `notebook-01`.

In [ ]:
flush()